# 01 — Profilage des sources SIRENE

**Objectif.** Comprendre les données *avant* de les transformer : schéma, types, valeurs manquantes, doublons,
états administratifs, nomenclatures, dates, géographie et cas difficiles réels.

**Ce que fait ce notebook :** lecture seule des fichiers bruts de `data/raw/` (Parquet) et du COG.
**Ce qu'il ne fait pas :** aucune transformation, aucune écriture. La logique vit dans `src/france_business_dynamics/`
(`validation/profile.py`) — le notebook ne fait que l'appeler et l'illustrer.

**Prérequis.** Les fichiers sources officiels ont été téléchargés dans `data/raw/` et le COG dans `data/reference/` (README, section *Téléchargement*).

In [ ]:

import matplotlib.pyplot as plt
import polars as pl

from france_business_dynamics.utils.config import SOURCE_FILES, ProjectPaths

pl.Config.set_tbl_rows(25)
pl.Config.set_fmt_str_lengths(60)
plt.rcParams.update({"figure.figsize": (9, 4.5), "axes.spines.top": False, "axes.spines.right": False})

PATHS = ProjectPaths.from_env()
print("Répertoire de données :", PATHS.data)

from france_business_dynamics.validation import profile
from france_business_dynamics.validation.schemas import ETABLISSEMENT_COLUMNS, UNITE_LEGALE_COLUMNS

# Profilage rapide sur les N premières lignes (None = fichier entier). Sur le stock complet
# (44 M de lignes), commencer par exemple avec 2_000_000 puis relancer sans limite.
SAMPLE_ROWS = None

def load(key, columns=None):
    """LazyFrame du Parquet brut `key` (colonnes sélectionnées tôt), ou None si le fichier est absent."""
    path = SOURCE_FILES[key].local_path(PATHS)
    if not path.exists():
        print(f"⚠ {key} absent ({path.name}) — section ignorée. Voir README, section « Téléchargement ».")
        return None
    frame = pl.scan_parquet(path)
    if columns:
        frame = frame.select([c for c in columns if c in frame.collect_schema()])
    return frame.head(SAMPLE_ROWS) if SAMPLE_ROWS else frame

et = load("stock_etablissement", ETABLISSEMENT_COLUMNS)
ul = load("stock_unite_legale", UNITE_LEGALE_COLUMNS)
print("Échantillon :", f"{SAMPLE_ROWS:,} lignes" if SAMPLE_ROWS else "fichier entier")

## 1. Vue d'ensemble des fichiers (métadonnées Parquet, sans charger les données)

In [ ]:
rows = []
for key in ("stock_unite_legale", "stock_etablissement", "stock_unite_legale_historique", "stock_etablissement_historique"):
    path = SOURCE_FILES[key].local_path(PATHS)
    if path.exists():
        info = profile.parquet_overview(path)
        rows.append({"source": key, "lignes": info["rows"], "colonnes": len(info["columns"]),
                     "groupes_de_lignes": info["row_groups"], "taille_Mo": info["file_mb"]})
display(pl.DataFrame(rows) if rows else "Aucun fichier brut trouvé.")

## 2. Colonnes retenues vs colonnes présentes

Seules les colonnes nécessaires à l'analyse sont conservées (aucune « au cas où »). Une colonne **MANQUANTE**
signifierait que l'Insee a modifié le schéma : le pipeline s'arrêterait avec un message explicite
(`validation/schemas.py::require_columns`).

In [ ]:
for label, frame_key, expected in (("Établissements", "stock_etablissement", ETABLISSEMENT_COLUMNS),
                                   ("Unités légales", "stock_unite_legale", UNITE_LEGALE_COLUMNS)):
    path = SOURCE_FILES[frame_key].local_path(PATHS)
    if path.exists():
        print(label)
        display(profile.schema_comparison(pl.scan_parquet(path), expected))

## 3. Valeurs manquantes et chaînes vides

In [ ]:
if et is not None:
    nulls = profile.null_profile(et)
    display(nulls)
    top = nulls.filter(pl.col("pct_manquant") > 0).head(10)
    if top.height:
        plt.barh(top["colonne"].to_list()[::-1], top["pct_manquant"].to_list()[::-1], color="#c0392b")
        plt.xlabel("% de lignes manquantes"); plt.title("Établissements — colonnes les plus incomplètes")
        plt.tight_layout(); plt.show()
    else:
        print("Aucune valeur manquante dans les colonnes retenues.")

## 4. Doublons sur les clés

`siret` doit être unique dans le stock des établissements, `siren` dans celui des unités légales.
Un excès de lignes ferait échouer le quality gate (`etablissement.siret_unique`).

In [ ]:
for frame, key in ((et, "siret"), (ul, "siren")):
    if frame is not None:
        print(profile.duplicate_profile(frame, key))

## 5. États administratifs

In [ ]:
if et is not None:
    dist = profile.value_distribution(et, "etatAdministratifEtablissement")
    display(dist)
    plt.bar(dist["etatAdministratifEtablissement"].fill_null("(vide)").to_list(), dist["lignes"].to_list(), color="#2c3e50")
    plt.title("Établissements par état administratif (A = actif, F = fermé)"); plt.ylabel("lignes"); plt.show()
if ul is not None:
    display(profile.value_distribution(ul, "etatAdministratifUniteLegale"))

## 6. Nomenclatures d'activité

`NAFRev2` est la nomenclature de l'analyse sectorielle. `NAFRev1`, `NAF1993` et `NAP` apparaissent dans les données
mais **ne sont jamais mélangées** à la NAF Rev.2 (leurs divisions n'ont pas le même sens). La NAF 2025 est une
colonne séparée, conservée telle quelle.

In [ ]:
if et is not None:
    display(profile.value_distribution(et, "nomenclatureActivitePrincipaleEtablissement"))
if ul is not None:
    display(profile.value_distribution(ul, "nomenclatureActivitePrincipaleUniteLegale"))

## 7. Tranches d'effectifs

In [ ]:
if et is not None:
    display(profile.value_distribution(et, "trancheEffectifsEtablissement", top_n=30))

## 8. Dates de création

Une date renseignée mais illisible est **conservée et signalée** (jamais supprimée). La date de création *enregistrée*
n'est pas la « création d'entreprise » des statistiques officielles de l'Insee.

In [ ]:
for frame, column in ((et, "dateCreationEtablissement"), (ul, "dateCreationUniteLegale")):
    if frame is not None:
        print(profile.date_profile(frame, column))

## 9. Géographie

Les codes commune sont des **chaînes** : `2A001` (Corse-du-Sud) et `01004` (zéro initial) seraient détruits par une
conversion entière.

In [ ]:
if et is not None:
    geo = profile.geography_profile(et)
    display(pl.DataFrame({"indicateur": list(geo), "valeur": [int(v) for v in geo.values()]}))
cog_path = SOURCE_FILES["cog_communes_2026"].local_path(PATHS)
if cog_path.exists():
    from france_business_dynamics.transformation.geography import read_cog_2026
    cog = read_cog_2026(cog_path).collect()
    print("COG 2026 :", cog.height, "communes ; types :", cog["type_commune"].unique().sort().to_list())
    print("Schéma COG (tout en chaînes) :", dict(cog.schema))

## 10. Cas difficiles réels (exemples)

In [ ]:
if et is not None:
    cases = {
        "Commune corse (2A/2B)": pl.col("codeCommuneEtablissement").str.starts_with("2A") | pl.col("codeCommuneEtablissement").str.starts_with("2B"),
        "Nomenclature historique": pl.col("nomenclatureActivitePrincipaleEtablissement").is_in(["NAFRev1", "NAF1993", "NAP"]),
        "Sans code commune": pl.col("codeCommuneEtablissement").is_null(),
        "Tranche d'effectifs absente": pl.col("trancheEffectifsEtablissement").is_null(),
    }
    cols = ["siret", "codeCommuneEtablissement", "activitePrincipaleEtablissement",
            "nomenclatureActivitePrincipaleEtablissement", "trancheEffectifsEtablissement", "etatAdministratifEtablissement"]
    for title, predicate in cases.items():
        print(title)
        display(profile.sample_rows(et, predicate, n=3, columns=cols))

## 11. Historiques (périodes de validité)

Une ligne = une *période*. L'indicateur `changement…` marque le changement par rapport à la période précédente.
Sur 72 M / 96 M de lignes, utiliser `SAMPLE_ROWS`.

In [ ]:
for key, state, change in (("stock_unite_legale_historique", "etatAdministratifUniteLegale", "changementEtatAdministratifUniteLegale"),
                           ("stock_etablissement_historique", "etatAdministratifEtablissement", "changementEtatAdministratifEtablissement")):
    frame = load(key, [state, change, "dateDebut", "dateFin"])
    if frame is not None:
        print(key)
        display(frame.group_by(state, change).len(name="periodes").sort("periodes", descending=True).collect())

## 12. Synthèse — problèmes de qualité repérés dans les sources

In [ ]:
findings = []
if et is not None:
    d = profile.date_profile(et, "dateCreationEtablissement")
    g = profile.geography_profile(et)
    n = d["rows"] or 1
    findings += [
        {"source": "établissements", "constat": "dates de création illisibles", "lignes": d["invalid"], "pct": round(d["invalid"] / n * 100, 4)},
        {"source": "établissements", "constat": "dates de création absentes", "lignes": d["absent"], "pct": round(d["absent"] / n * 100, 4)},
        {"source": "établissements", "constat": "dates de création avant 1900", "lignes": d["before_1900"], "pct": round(d["before_1900"] / n * 100, 6)},
        {"source": "établissements", "constat": "dates de création après le snapshot", "lignes": d["after_snapshot"], "pct": round(d["after_snapshot"] / n * 100, 6)},
        {"source": "établissements", "constat": "sans code commune", "lignes": g["no_commune"], "pct": round(g["no_commune"] / n * 100, 4)},
        {"source": "établissements", "constat": "code commune au format inattendu", "lignes": g["bad_format"], "pct": round(g["bad_format"] / n * 100, 4)},
    ]
if ul is not None:
    d = profile.date_profile(ul, "dateCreationUniteLegale")
    n = d["rows"] or 1
    findings += [
        {"source": "unités légales", "constat": "dates de création illisibles", "lignes": d["invalid"], "pct": round(d["invalid"] / n * 100, 4)},
        {"source": "unités légales", "constat": "dates de création absentes", "lignes": d["absent"], "pct": round(d["absent"] / n * 100, 4)},
        {"source": "unités légales", "constat": "dates de création avant 1900", "lignes": d["before_1900"], "pct": round(d["before_1900"] / n * 100, 6)},
        {"source": "unités légales", "constat": "dates de création après le snapshot", "lignes": d["after_snapshot"], "pct": round(d["after_snapshot"] / n * 100, 6)},
    ]
display(pl.DataFrame(findings) if findings else "Rien à synthétiser (sources absentes).")
